In [1]:
import os
import warnings

# from google.colab import drive
# drive.mount('/content/drive')
# base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
# os.chdir(base_path)

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import LambdaCallback, EarlyStopping, ModelCheckpoint

import tensorflow as tf

from tensorflow import keras
import keras_hub
from tensorflow.keras import regularizers, layers
from tensorflow.keras.layers import Input, Dropout, Dense, Layer, Embedding, Lambda
from keras_hub.layers import PositionEmbedding
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, GlobalMaxPooling1D, GlobalAveragePooling1D, TextVectorization, BatchNormalization
from tensorflow.keras.models import Model, Sequential

import yfinance as yf
from config import config

from dataclasses import dataclass
import glob
from pprint import pprint
import nlpaug.augmenter.word as naw
from pygooglenews import GoogleNews

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings("ignore", category=UserWarning, module="keras")

In [2]:
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape("!#$%&'()*+,-./:;<=>?@\\^_`{|}~"), ""
    )

def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[MASK]"]):
    vectorize_layer = TextVectorization(
        max_tokens = vocab_size,
        output_mode = "int",
        standardize = custom_standardization,
        output_sequence_length = max_seq
    )
    vectorize_layer.adapt(texts)

    vocab = vectorize_layer.get_vocabulary()
    vocab = vocab[2: vocab_size - len(special_tokens)] + ["[mask]"]
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

def encode(texts):
    encoded_texts = vectorize_layer(texts)
    return encoded_texts.numpy()

def get_masked_input_and_labels(encoded_texts):
    inp_mask = np.random.rand(*encoded_texts.shape) < 0.15
    inp_mask[encoded_texts <= 2] = False
    labels = -1 * np.ones(encoded_texts.shape, dtype = int)
    labels[inp_mask] = encoded_texts[inp_mask]

    encoded_texts_masked = np.copy(encoded_texts)
    inp_mask_2mask = inp_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    encoded_texts_masked[inp_mask_2mask] = (mask_token_id)

    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1/9)
    encoded_texts_masked[inp_mask_2random] = np.random.randint(3, mask_token_id, inp_mask_2random.sum())

    sample_weights = np.ones(labels.shape)
    sample_weights[labels == -1] = 0

    y_labels = np.copy(encoded_texts)

    return encoded_texts_masked, y_labels, sample_weights

In [3]:
# # Dataset for MLM Pretraining For Google Colab:
# news_raw = pd.read_csv(os.path.join("data", "abcnews-date-text.csv"))
# news_text_raw = news_raw["headline_text"]

# vectorize_layer = get_vectorize_layer(
#     news_text_raw.tolist(),
#     config.VOCAB_SIZE,
#     config.MAX_LEN,
#     special_tokens=["[mask]"],
# )
# mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]

# # mlm_ds was created on a local machine, then added to the drive
# mlm_ds = tf.data.Dataset.load(os.path.join("data", "mlm_dataset_final"))
# mlm_ds_small = mlm_ds.shard(num_shards=12, index=0)

# # Dataset For Sentiment Classification
# sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header = None)
# sentiment_raw.columns = ["Output", "Input"]
# sentiment_dataset = sentiment_raw[["Input", "Output"]]

# df_0 = sentiment_dataset[sentiment_dataset["Output"] == "negative"]
# df_1 = sentiment_dataset[sentiment_dataset["Output"] == "neutral"]
# df_2 = sentiment_dataset[sentiment_dataset["Output"] == "positive"]

# min_label = min(len(df_0), len(df_1), len(df_2))
# df_0_downsampled = resample(df_0, replace=False, n_samples=min_label, random_state=42)
# df_1_downsampled = resample(df_1, replace=False, n_samples=min_label, random_state=42)
# df_2_downsampled = resample(df_2, replace=False, n_samples=min_label, random_state=42)

# sentiment_dataset_balanced = pd.concat([df_0_downsampled, df_1_downsampled, df_2_downsampled])

# X, y = sentiment_dataset_balanced['Input'], sentiment_dataset_balanced['Output']
# X_encoded = encode(X)

# le = LabelEncoder()
# y_encoded = le.fit_transform(y)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_encoded, y_encoded, test_size=0.1, random_state=42
# )

# train_classifier_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(config.BATCH_SIZE))
# test_classifier_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(config.BATCH_SIZE)

In [4]:
# Dataset for MLM Pretraining For Local Machine
news_raw = pd.read_csv(os.path.join("data", "abcnews-date-text.csv"))
news_text_raw = news_raw["headline_text"]

vectorize_layer = get_vectorize_layer(
    news_text_raw.tolist(),
    config.VOCAB_SIZE,
    config.MAX_LEN,
    special_tokens=["[mask]"],
)

mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]

x_all_encoded = encode(news_text_raw)

x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(x_all_encoded)

mlm_ds = tf.data.Dataset.from_tensor_slices(
    (x_masked_train, y_masked_labels, sample_weights)
)
mlm_ds = mlm_ds.shuffle(1000).batch(config.BATCH_SIZE)
mlm_ds_small = mlm_ds.shard(num_shards=256, index=0)

In [5]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng') # 에러 메시지에서 요구한 파일
nltk.download('punkt_tab') # 문장 토큰화를 위해 가끔 요구됨

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/juhyeongpang/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/juhyeongpang/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/juhyeongpang/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juhyeongpang/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
# 1. 데이터 로드 및 전처리
sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header=None)
sentiment_raw.columns = ["Output", "Input"]

# 클래스별 분리
df_neg = sentiment_raw[sentiment_raw["Output"] == "negative"]
df_neu = sentiment_raw[sentiment_raw["Output"] == "neutral"]
df_pos = sentiment_raw[sentiment_raw["Output"] == "positive"]

# 2. 다운샘플링 (최소 개수에 맞춤)
min_label = min(len(df_neg), len(df_neu), len(df_pos))
df_neg_down = resample(df_neg, replace=False, n_samples=min_label, random_state=42)
df_neu_down = resample(df_neu, replace=False, n_samples=min_label, random_state=42)
df_pos_down = resample(df_pos, replace=False, n_samples=min_label, random_state=42)

# 3. nlpaug 증강 설정
aug = naw.SynonymAug(aug_src='wordnet')

def augment_text(df, target_count):
    current_count = len(df)
    if current_count >= target_count:
        return df
    
    aug_samples = []
    needed = target_count - current_count
    label_name = df.iloc[0]['Output']
    print(f"Augmenting '{label_name}' class: {current_count} -> {target_count}...")
    
    while len(aug_samples) < needed:
        for text in df['Input']:
            if len(aug_samples) >= needed:
                break
            # 증강 실행
            augmented_text = aug.augment(text)[0]
            aug_samples.append(augmented_text)
            
    df_aug = pd.DataFrame({'Input': aug_samples, 'Output': [label_name] * len(aug_samples)})
    return pd.concat([df, df_aug])

# 4. 600개 -> 2000개로 뻥튀기
target_n = 2000 
df_neg_final = augment_text(df_neg_down, target_n)
df_neu_final = augment_text(df_neu_down, target_n)
df_pos_final = augment_text(df_pos_down, target_n)

# 5. 합치기 및 셔플
final_df = pd.concat([df_neg_final, df_neu_final, df_pos_final]).sample(frac=1, random_state=42).reset_index(drop=True)

# 6. 인코딩 및 데이터셋 생성
# 주의: encode 함수는 본인의 환경에 정의되어 있어야 합니다.
X_encoded = encode(final_df['Input']) 

le = LabelEncoder()
y_encoded = le.fit_transform(final_df['Output'])

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.1, random_state=42
)

# TF 데이터셋 구성
train_classifier_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
                       .shuffle(len(X_train))
                       .batch(32)) # BATCH_SIZE는 상황에 맞게 조절

test_classifier_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

print("Done! 데이터 준비가 완료되었습니다.")

Augmenting 'negative' class: 604 -> 2000...
Augmenting 'neutral' class: 604 -> 2000...
Augmenting 'positive' class: 604 -> 2000...
Done! 데이터 준비가 완료되었습니다.


In [7]:
mlm_ds_small.element_spec

(TensorSpec(shape=(None, 16), dtype=tf.int64, name=None),
 TensorSpec(shape=(None, 16), dtype=tf.int64, name=None),
 TensorSpec(shape=(None, 16), dtype=tf.float64, name=None))

In [16]:
def bert_module(query, key, value, i, mask=None):
    # Multi headed self-attention
    attention_output = layers.MultiHeadAttention(
        num_heads=config.NUM_HEAD,
        key_dim=config.EMBED_DIM // config.NUM_HEAD,
        name="encoder_{}_multiheadattention".format(i),
    )(query, key, value, attention_mask=mask)
    attention_output = layers.Dropout(0.1, name="encoder_{}_att_dropout".format(i))(
        attention_output
    )
    attention_output = layers.LayerNormalization(
        epsilon=1e-6, name="encoder_{}_att_layernormalization".format(i)
    )(query + attention_output)

    # Feed-forward layer
    ffn = keras.Sequential(
        [
            layers.Dense(config.FF_DIM, activation="relu"),
            layers.Dense(config.EMBED_DIM),
        ],
        name="encoder_{}_ffn".format(i),
    )
    ffn_output = ffn(attention_output)
    ffn_output = layers.Dropout(0.1, name="encoder_{}_ffn_dropout".format(i))(
        ffn_output
    )
    sequence_output = layers.LayerNormalization(
        epsilon=1e-6, name="encoder_{}_ffn_layernormalization".format(i)
    )(attention_output + ffn_output)
    return sequence_output


loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction=None)
loss_tracker = keras.metrics.Mean(name="loss")


class MaskedLanguageModel(keras.Model):

    def compute_loss(self, x=None, y=None, y_pred=None, sample_weight=None):

        loss = loss_fn(y, y_pred, sample_weight)
        loss_tracker.update_state(loss, sample_weight=sample_weight)
        return keras.ops.sum(loss)

    def compute_metrics(self, x, y, y_pred, sample_weight):

        # Return a dict mapping metric names to current value
        return {"loss": loss_tracker.result()}

    @property
    def metrics(self):
        # We list our `Metric` objects here so that `reset_states()` can be
        # called automatically at the start of each epoch
        # or at the start of `evaluate()`.
        # If you don't implement this property, you have to call
        # `reset_states()` yourself at the time of your choosing.
        return [loss_tracker]


def create_masked_language_bert_model():
    inputs = layers.Input(shape=(config.MAX_LEN,), name="Input")
    
    word_embeddings = layers.Embedding(
        config.VOCAB_SIZE, 
        config.EMBED_DIM, 
        mask_zero=True, 
        name="word_embedding"
    )(inputs)
    position_embeddings = keras_hub.layers.PositionEmbedding(
        sequence_length=config.MAX_LEN,
        name="position_embedding"
    )(word_embeddings)
    embeddings = word_embeddings + position_embeddings

    encoder_output = embeddings
    # for i in range(1, config.NUM_LAYERS+1):
    for i in range(1, 4+1):
        encoder_output = bert_module(encoder_output, encoder_output, encoder_output, i)

    mlm_output = layers.Dense(config.VOCAB_SIZE, name="mlm_cls", activation="softmax")(
        encoder_output
    )
    mlm_model = MaskedLanguageModel(inputs, mlm_output, name="masked_bert_model")

    optimizer = keras.optimizers.Adam(learning_rate=config.LR)
    mlm_model.compile(optimizer=optimizer)
    return mlm_model


id2token = dict(enumerate(vectorize_layer.get_vocabulary()))
token2id = {y: x for x, y in id2token.items()}


class MaskedTextGenerator(keras.callbacks.Callback):
    def __init__(self, sample_tokens, top_k=5):
        self.sample_tokens = sample_tokens
        self.k = top_k

    def decode(self, tokens):
        return " ".join([id2token[t] for t in tokens if t != 0])

    def convert_ids_to_tokens(self, id):
        return id2token[id]

    def on_epoch_end(self, epoch, logs=None):
        prediction = self.model.predict(self.sample_tokens)

        masked_index = np.where(self.sample_tokens == mask_token_id)
        masked_index = masked_index[1]
        mask_prediction = prediction[0][masked_index]

        top_indices = mask_prediction[0].argsort()[-self.k :][::-1]
        values = mask_prediction[0][top_indices]

        for i in range(len(top_indices)):
            p = top_indices[i]
            v = values[i]
            tokens = np.copy(sample_tokens[0])
            tokens[masked_index[0]] = p
            result = {
                "input_text": self.decode(sample_tokens[0].numpy()),
                "prediction": self.decode(tokens),
                "probability": v,
                "predicted mask token": self.convert_ids_to_tokens(p),
            }
            pprint(result)


sample_tokens = vectorize_layer(["Google wont [mask] replacing our news headlines with terrible AI"])

bert_masked_model = create_masked_language_bert_model()
bert_masked_model.summary()

Model: "masked_bert_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input (InputLayer)  │ (None, 16)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 16, 128)   │  3,840,000 │ Input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, 16, 128)   │      2,048 │ word_embedding[0… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 16, 128)   │          0 │ word_embedding[0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_multihea… │ (None, 16, 128)   │     66,048 │ add_9[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_9[0][0],      │
│                     │                   │            │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_drop… │ (None, 16, 128)   │          0 │ encoder_1_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 16, 128)   │          0 │ add_9[0][0],      │
│                     │                   │            │ encoder_1_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_laye… │ (None, 16, 128)   │        256 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn       │ (None, 16, 128)   │    131,712 │ encoder_1_att_la… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_drop… │ (None, 16, 128)   │          0 │ encoder_1_ffn[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 16, 128)   │          0 │ encoder_1_att_la… │
│                     │                   │            │ encoder_1_ffn_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_laye… │ (None, 16, 128)   │        256 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_multihea… │ (None, 16, 128)   │     66,048 │ encoder_1_ffn_la… │
│ (MultiHeadAttentio… │                   │            │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_1_ffn_la… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_drop… │ (None, 16, 128)   │          0 │ encoder_2_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 16, 128)   │          0 │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_2_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_laye… │ (None, 16, 128)   │        256 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 8,505,136 (32.44 MB)

 Trainable params: 8,505,136 (32.44 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
class MaskedTextGenerator(keras.callbacks.Callback):
    def __init__(self, sample_tokens, top_k=5):
        self.sample_tokens = sample_tokens
        self.k = top_k

    def decode(self, tokens):
        return " ".join([id2token[t] for t in tokens if t != 0])

    def convert_ids_to_tokens(self, id):
        return id2token[id]

    def on_epoch_end(self, epoch, logs=None):
        prediction = self.model.predict(self.sample_tokens)

        masked_index = np.where(self.sample_tokens == mask_token_id)
        masked_index = masked_index[1]
        mask_prediction = prediction[0][masked_index]

        top_indices = mask_prediction[0].argsort()[-self.k :][::-1]
        values = mask_prediction[0][top_indices]

        for i in range(len(top_indices)):
            p = top_indices[i]
            v = values[i]
            tokens = np.copy(sample_tokens[0])
            tokens[masked_index[0]] = p
            result = {
                "input_text": self.decode(sample_tokens[0].numpy()),
                "prediction": self.decode(tokens),
                "probability": v,
                "predicted mask token": self.convert_ids_to_tokens(p),
            }
            pprint(result)
generator_callback = MaskedTextGenerator(sample_tokens.numpy())

earlyStopping_callback = EarlyStopping(
    monitor="loss",
    min_delta=0.005,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
)

checkpoint_callback = ModelCheckpoint(
    os.path.join("models", "model_4L_weights_cp_best.weights.h5"), 
    save_best_only=True,
    monitor="loss",
    save_weights_only=True,
    mode="min",
    verbose=1
)

In [20]:
def save_model_weights(model, file_name, folder_name):
    os.makedirs(folder_name, exist_ok=True)

    words = file_name.split(".")

    model_name = words[0]

    existing_files = [f for f in os.listdir(folder_name) if f.startswith(model_name)]
    next_number = len(existing_files) + 1
    words[0] = f"{model_name}_{next_number}"

    file_name = ".".join(words)
    save_path = os.path.join(folder_name, file_name)

    model.save_weights(save_path)
    print(f"model saved to: {save_path}")

In [21]:
bert_masked_model.load_weights(os.path.join("models", "model_4L_weights_cp_best.weights.h5"))
# bert_masked_model.fit(mlm_ds, epochs=50, callbacks=[generator_callback, checkpoint_callback, earlyStopping_callback])
# save_model_weights(bert_masked_model, "bert_masked_model_v2.weights.h5", "models")
# bert_masked_model.save("BERT_Pretrained.keras")

In [13]:
bert_masked_model = keras.models.load_model(
    os.path.join("models", "BERT_Pretrained.keras"), 
    custom_objects={"MaskedLanguageModel": MaskedLanguageModel}
)

In [22]:
bert_masked_model.summary()

Model: "masked_bert_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input (InputLayer)  │ (None, 16)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 16, 128)   │  3,840,000 │ Input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, 16, 128)   │      2,048 │ word_embedding[0… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 16, 128)   │          0 │ word_embedding[0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_multihea… │ (None, 16, 128)   │     66,048 │ add_9[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_9[0][0],      │
│                     │                   │            │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_drop… │ (None, 16, 128)   │          0 │ encoder_1_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 16, 128)   │          0 │ add_9[0][0],      │
│                     │                   │            │ encoder_1_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_laye… │ (None, 16, 128)   │        256 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn       │ (None, 16, 128)   │    131,712 │ encoder_1_att_la… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_drop… │ (None, 16, 128)   │          0 │ encoder_1_ffn[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 16, 128)   │          0 │ encoder_1_att_la… │
│                     │                   │            │ encoder_1_ffn_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_laye… │ (None, 16, 128)   │        256 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_multihea… │ (None, 16, 128)   │     66,048 │ encoder_1_ffn_la… │
│ (MultiHeadAttentio… │                   │            │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_1_ffn_la… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_drop… │ (None, 16, 128)   │          0 │ encoder_2_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 16, 128)   │          0 │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_2_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_laye… │ (None, 16, 128)   │        256 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 8,505,136 (32.44 MB)

 Trainable params: 8,505,136 (32.44 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
pretrained_bert_model = keras.Model(
    bert_masked_model.input, bert_masked_model.get_layer("encoder_4_ffn_layernormalization").output
)
pretrained_bert_model.trainable = False

pretrained_bert_model.summary()

def create_classifier_bert_model():
    inputs = layers.Input((config.MAX_LEN,), dtype="int64")
    sequence_output = pretrained_bert_model(inputs)
    
    pooled_output = layers.Lambda(lambda x: x[:, 0, :])(sequence_output)
    x = layers.BatchNormalization()(pooled_output)
    
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x) 
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(64, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    outputs = layers.Dense(3, activation="softmax")(x)

    classifier_model = Model(inputs, outputs, name="classification")
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5) 
    
    classifier_model.compile(
        optimizer=optimizer, 
        loss="sparse_categorical_crossentropy", 
        metrics=["accuracy"]
    )
    return classifier_model


classifer_model = create_classifier_bert_model()
classifer_model.summary()

classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)

pretrained_bert_model.trainable = True
optimizer = keras.optimizers.Adam()

classifer_model.compile(
    optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input (InputLayer)  │ (None, 16)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 16, 128)   │  3,840,000 │ Input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, 16, 128)   │      2,048 │ word_embedding[0… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 16, 128)   │          0 │ word_embedding[0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_multihea… │ (None, 16, 128)   │     66,048 │ add_9[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_9[0][0],      │
│                     │                   │            │ add_9[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_drop… │ (None, 16, 128)   │          0 │ encoder_1_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 16, 128)   │          0 │ add_9[0][0],      │
│                     │                   │            │ encoder_1_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_laye… │ (None, 16, 128)   │        256 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn       │ (None, 16, 128)   │    131,712 │ encoder_1_att_la… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_drop… │ (None, 16, 128)   │          0 │ encoder_1_ffn[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 16, 128)   │          0 │ encoder_1_att_la… │
│                     │                   │            │ encoder_1_ffn_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_laye… │ (None, 16, 128)   │        256 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_multihea… │ (None, 16, 128)   │     66,048 │ encoder_1_ffn_la… │
│ (MultiHeadAttentio… │                   │            │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_1_ffn_la… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_drop… │ (None, 16, 128)   │          0 │ encoder_2_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 16, 128)   │          0 │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_2_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_laye… │ (None, 16, 128)   │        256 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,635,136 (17.68 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,635,136 (17.68 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional_12 (Functional)      │ (None, 16, 128)        │     4,635,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,661,379 (17.78 MB)

 Trainable params: 25,603 (100.01 KB)

 Non-trainable params: 4,635,776 (17.68 MB)

Epoch 1/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.3393 - loss: 1.4903 - val_accuracy: 0.3267 - val_loss: 1.3062
Epoch 2/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.3531 - loss: 1.3986 - val_accuracy: 0.3483 - val_loss: 1.2598
Epoch 3/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.3694 - loss: 1.3607 - val_accuracy: 0.3783 - val_loss: 1.2355
Epoch 4/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.3669 - loss: 1.3502 - val_accuracy: 0.3850 - val_loss: 1.2074
Epoch 5/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.3767 - loss: 1.3107 - val_accuracy: 0.3983 - val_loss: 1.1883
Epoch 1/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 38s 145ms/step - accuracy: 0.4363 - loss: 1.1440 - val_accuracy: 0.3700 - val_loss: 1.8472
Epoch 2/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 25s 146ms/step - accuracy: 0.6985 - loss: 0.6945 - val_accuracy: 0.6183 - val_loss: 1.4992
Epoch 3/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 19s 112ms/step - accuracy: 0.8683 - loss: 0.3384 - val_accur

In [24]:
classifer_model.save_weights("march_sixth_4Layers.weights.h5")

In [25]:
def predict(text, model):
    text_encoded = encode(text)
    # print("Encoded Text: ", text_encoded)
    
    pred = model(text_encoded, training=False)

    probabilities = tf.nn.softmax(pred, axis=-1).numpy()[0]

    pred_class = np.argmax(probabilities)
    sentiment = le.inverse_transform([pred_class])[0]
    confidence_score = probabilities[pred_class]

    class_names = le.classes_
    all_probs = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}
    
    return sentiment, confidence_score, all_probs

def predict_and_print(text, model):
    new_texts = [text]
    result, confidence, all_probs = predict(new_texts, model)
    
    print(f"Input Headline: \"{text}\"")
    print(f"Top Prediction: {result} ({confidence * 100:.2f}%)")
    print("-" * 30)
    print("Class Probabilities:")
    
    sorted_probs = sorted(all_probs.items(), key=lambda item: item[1], reverse=True)
    
    for label, prob in sorted_probs:
        bar = "#" * int(prob * 20) 
        print(f" - {label:<12}: {prob * 100:>6.2f}% {bar}")

    return result

In [26]:
test_headlines = [
    # 1. 명확한 부정 (Negative)
    "Apple shares plummet after disappointing iPhone sales report",
    "Lawsuit filed against Apple over battery throttling issues",
    "Supply chain disruptions expected to delay new Mac releases",
    
    # 2. 명확한 긍정 (Positive)
    "Apple's quarterly profit exceeds analyst expectations",
    "Revolutionary AI features announced for upcoming iOS update",
    "Warren Buffett increases stake in Apple, citing strong ecosystem",
    
    # 3. 중립 또는 단순 정보 (Neutral/Info)
    "Apple to hold its annual developer conference in June",
    "New software update available for Apple Watch users",
    "Tim Cook delivers keynote speech at university graduation",
    
    # 4. 모델을 시험하는 모호한 문장 (Edge Cases)
    "Apple is down today, but long-term outlook remains bright", # 부정+긍정 혼합
    "The new iPhone is not as bad as critics predicted",          # 이중 부정
    "Apple's competitor Samsung sees record-breaking growth",    # 경쟁사 소식 (애플에겐 위협?)
    "Is Apple losing its innovative edge?",                      # 의문문형 부정
]

In [28]:
for headline in test_headlines:
    print(encode(headline))
    predict_and_print(headline, classifer_model)
    print("\n" + "="*50 + "\n")

[1879  637 8661   12 4977 7374  695   52    0    0    0    0    0    0
    0    0]
Input Headline: "Apple shares plummet after disappointing iPhone sales report"
Top Prediction: neutral (46.39%)
------------------------------
Class Probabilities:
 - neutral     :  46.39% #########
 - negative    :  30.54% ######
 - positive    :  23.07% ####


[ 3762 15027    45  1879     8  5033     1   631     0     0     0     0
     0     0     0     0]
Input Headline: "Lawsuit filed against Apple over battery throttling issues"
Top Prediction: neutral (57.48%)
------------------------------
Class Probabilities:
 - neutral     :  57.48% ###########
 - positive    :  21.28% ####
 - negative    :  21.24% ####


[1071 5216 8391  419    2  731   13 8233 1493    0    0    0    0    0
    0    0]
Input Headline: "Supply chain disruptions expected to delay new Mac releases"
Top Prediction: neutral (57.53%)
------------------------------
Class Probabilities:
 - neutral     :  57.53% ###########
 - positive

In [29]:
gn = GoogleNews(lang = "en", country = "US")
google_news = gn.search('Apple', when='7d')

In [30]:
google_news

{'feed': {'generator_detail': {'name': 'NFE/5.0'},
  'generator': 'NFE/5.0',
  'title': '"Apple when:7d" - Google News',
  'title_detail': {'type': 'text/plain',
   'language': None,
   'base': '',
   'value': '"Apple when:7d" - Google News'},
  'links': [{'rel': 'alternate',
    'type': 'text/html',
    'href': 'https://news.google.com/search?q=Apple+when:7d&ceid=US:en&hl=en-US&gl=US'}],
  'link': 'https://news.google.com/search?q=Apple+when:7d&ceid=US:en&hl=en-US&gl=US',
  'language': 'en-US',
  'publisher': 'news-webmaster@google.com',
  'publisher_detail': {'email': 'news-webmaster@google.com'},
  'rights': 'Copyright © 2026 Google. All rights reserved. This XML feed is made available solely for the purpose of rendering Google News results within a personal feed reader for personal, non-commercial use. Any other use of the feed is expressly prohibited. By accessing this feed or using these results in any manner whatsoever, you agree to be bound by the foregoing restrictions.',
  'r

In [31]:
for entry in google_news['entries']:
    print(entry['title'])

Apple introduces iPhone 17e - Apple
I’m most excited about Apple’s affordable MacBook, with one concern - 9to5Mac
Apple Is in Its Affordable Era. Sort Of. - The New York Times
MacBook Neo hands-on: Apple build quality at a substantially lower price - Ars Technica
Apple Just Launched an Entire Lineup of AI-Powered Macs, iPhones, and iPads - eWeek
Apple launches lower cost iPhone 17e and a new iPad Air powered by its M4 chip - CNBC
Apple debuts $599 iPhone 17e, more powerful iPad Airs - Yahoo Finance
A new cheaper MacBook just dropped — here’s everything you need to know - NBC News
A Remarkable Turnaround: Apple’s Back And Better Than Ever With The MacBook Neo - Medium
7 Apple Releases You May Have Missed This Week - Gear Patrol
Charting the vibes in the 2025 Apple Report Card - Six Colors
Say hello to MacBook Neo - Apple
Apple raises MacBook prices across the board as M5 chips, new displays signal AI-first strategy - CNBC
Apple introduces the new iPad Air, powered by M4 - Apple
Apple de

In [32]:
len(google_news['entries'])

100

In [33]:
positive_count = 0
negative_count = 0
neutral_count = 0

for entry in google_news['entries']:
    result = predict_and_print(entry['title'], classifer_model)
    if result is not None:
        if ("positive" in result):
            positive_count += 1
        elif ("negative" in result):
            negative_count += 1
        elif ("neutral" in result):
            neutral_count += 1

    print("\n" + "="*50 + "\n")

print(f"Positive Count: {positive_count}")
print(f"Negative Count: {negative_count}")
print(f"Neutral Count: {neutral_count}")

Input Headline: "Apple introduces iPhone 17e - Apple"
Top Prediction: positive (50.14%)
------------------------------
Class Probabilities:
 - positive    :  50.14% ##########
 - neutral     :  27.35% #####
 - negative    :  22.51% ####


Input Headline: "I’m most excited about Apple’s affordable MacBook, with one concern - 9to5Mac"
Top Prediction: negative (47.02%)
------------------------------
Class Probabilities:
 - negative    :  47.02% #########
 - positive    :  29.35% #####
 - neutral     :  23.63% ####


Input Headline: "Apple Is in Its Affordable Era. Sort Of. - The New York Times"
Top Prediction: neutral (57.55%)
------------------------------
Class Probabilities:
 - neutral     :  57.55% ###########
 - positive    :  21.24% ####
 - negative    :  21.21% ####


Input Headline: "MacBook Neo hands-on: Apple build quality at a substantially lower price - Ars Technica"
Top Prediction: negative (57.47%)
------------------------------
Class Probabilities:
 - negative    :  57.47% 